## Defining the dataframe 
- Due to the edition of Databricks, getting .csv data through spark is not doable. 
- Therefore, defining dataframe through pandas and convert it to spark dataframe through **_spark.createDataFrame()_**

In [0]:
# df = spark.read.option("header",True).option("inferSchema",True).csv("/Workspace/Users/hhshin9879@gmail.com/energy-consumption-pipeline/demo/global_energy_consumption.csv")
import pandas as pd

df = pd.read_csv("/Workspace/Users/hhshin9879@gmail.com/energy-consumption-pipeline/data/world_energy_consumption.csv")
spark_df = spark.createDataFrame(df)

In [0]:
display(spark_df)

### Showing columns

In [0]:
spark_df.columns

## Select the columns which are going to be used
- Country-level greenhouse gas emissions prediction
based on energy consumption and macroeconomic features
- Countries : [`United States`,`China`,`India`,`Sweden`]
- Lag features (`emissions_lag1`, `emissions_lag2`) were added to incorporate historical emission values, enabling the model to capture temporal trends. For example, in the case of China, before 2000, `greenhouse_gas_emissions` value does not exist.
- In the case `gdp`, from 2019 to 2022, the data does not exist for the countries in the scope.
- herefore, the data from the year of 2002 to 2018 will be in the scope. 

In [0]:
from pyspark.sql.functions import col

col_list = [
    "country",
    "year",
    "population",
    "gdp",
    "coal_consumption",
    "oil_consumption",
    "gas_consumption",
    "renewables_consumption",
    "greenhouse_gas_emissions"
]

df_clean = spark_df.select(*col_list)
display(df_clean)

In [0]:
countries = [
    "United States",
    "China",
    "India",
    "Sweden"
]

# The data ends at 2019
df_countries = (
    df_clean
    .filter(col("country").isin(countries))
    .filter((col("year") >= 2002) & (col("year") <= 2018))
    .dropna()
)

display(df_countries)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window = Window.partitionBy("country").orderBy("year")

df_countries_window = df_countries.withColumn("emissions_lag1", lag("greenhouse_gas_emissions", 1).over(window))
df_countries_window = df_countries_window.withColumn("emissions_lag2", lag("greenhouse_gas_emissions", 2).over(window))

df_countries_window = df_countries_window.dropna()
display(df_countries_window)

## Visualisation

In [0]:
cols_to_plot = [
    "country",
    "year",
    "gdp",
    "coal_consumption",
    "oil_consumption",
    "gas_consumption",
    "renewables_consumption",
    "greenhouse_gas_emissions"
]

df_viz = df_countries.select(*cols_to_plot).dropna()
pdf = df_viz.toPandas()


In [0]:
for country in countries:
    subset = pdf[pdf["country"] == country].sort_values("year").copy()
    
    # minmax standardisation
    for col in [
        "gdp",
        "coal_consumption",
        "oil_consumption",
        "gas_consumption",
        "renewables_consumption",
        "greenhouse_gas_emissions"
    ]:
        subset[col] = (subset[col] - subset[col].min()) / (subset[col].max() - subset[col].min())
    
    plt.figure()
    
    for col in [
        "gdp",
        "coal_consumption",
        "oil_consumption",
        "gas_consumption",
        "renewables_consumption",
        "greenhouse_gas_emissions"
    ]:
        plt.plot(subset["year"], subset[col], label=col)
    
    plt.title(f"{country}")
    plt.xlabel("Year")
    plt.ylabel("Scaled Value (0-1)")
    plt.legend()
    plt.show()

## Linear Regression

- After selecting the final set of features, the input variables were combined into a single feature vector using `VectorAssembler`.

- To improve numerical stability and ensure that variables with larger scales do not dominate the optimisation process, the assembled features were standardised using `StandardScaler`.

- A linear regression model was then trained to predict `greenhouse_gas_emissions`.  
Because the dataset has a temporal structure, the train and test sets were split by year instead of using a random split.

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Drop nulls
df_model = df_countries_window.dropna()

# 2. Define features and label
feature_cols = [
    "population",
    "gdp",
    "coal_consumption",
    "oil_consumption",
    "gas_consumption",
    "renewables_consumption",
    "emissions_lag1",
    "emissions_lag2"
]

label_col = "greenhouse_gas_emissions"

# 3. Assemble features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_assembled = assembler.transform(df_model)

# 4. Scale features
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

# 5. Time-based split
train_df = df_scaled.filter(col("year") <= 2015)
test_df = df_scaled.filter(col("year") > 2015)

# 6. Train model
lr = LinearRegression(
    featuresCol="scaled_features",
    labelCol=label_col,
    predictionCol="prediction"
)

lr_model = lr.fit(train_df)

# 7. Predict
predictions = lr_model.transform(test_df)

# 8. Evaluate
rmse_evaluator = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="r2"
)

rmse = rmse_evaluator.evaluate(predictions)
mae = mae_evaluator.evaluate(predictions)
r2 = r2_evaluator.evaluate(predictions)

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R2: {r2}")

# 9. Show predictions
display(
    predictions.select(
        "country",
        "year",
        label_col,
        "prediction"
    ).orderBy("country", "year")
)

# 10. Coefficients
print("Intercept:", lr_model.intercept)
for feature_name, coef in zip(feature_cols, lr_model.coefficients):
    print(f"{feature_name}: {coef}")